# 10장 실습 — 배차와 할당 문제

호출 여러 건이 동시에 들어왔을 때 어느 차를 누구에게 보낼지 정하는 문제입니다.
교재 10장에 대응합니다.

이 실습의 뼈대는 3장과 같습니다.
**느리지만 틀릴 데 없는 방법으로 정답을 만들고, 그것으로 빠른 방법을 검증합니다.**
여기서는 모든 경우를 세어 보는 완전탐색이 정답지이고, 헝가리안 알고리즘이 검증 대상입니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

## 1. 비용행렬 (교재 10.1)

행이 승객, 열이 차량입니다. 칸 하나가 "이 차가 이 승객에게 가는 데 걸리는 분"입니다.

In [ ]:
import numpy as np

from smartmob.teaching.dispatch import cost_matrix

passengers = [(37.539, 127.215), (37.545, 127.200), (37.552, 127.190)]
vehicles = [(37.540, 127.210), (37.560, 127.195), (37.535, 127.225)]

costs = cost_matrix(passengers, vehicles)
np.round(costs, 2)

## 2. 탐욕 배차 (교재 10.2)

먼저 부른 사람부터 남은 차 중 가장 가까운 것을 줍니다.
사람이 손으로 하는 방식이고, 배차 담당자가 실제로 이렇게 합니다.

In [ ]:
from smartmob.teaching.dispatch import greedy_match, optimal_match

g = greedy_match(costs)
for m in g.matches:
    print(f"승객 {m.passenger} ← 차량 {m.vehicle}  {m.cost:.2f}분")
print(f"합계 {sum(m.cost for m in g.matches):.2f}분")

## 3. 할당 문제 (교재 10.3)

전체 합이 가장 작아지도록 한꺼번에 정합니다.
먼저 부른 사람이 손해를 볼 수 있지만 전체는 좋아집니다.

In [ ]:
o = optimal_match(costs)
for m in o.matches:
    print(f"승객 {m.passenger} ← 차량 {m.vehicle}  {m.cost:.2f}분")
print(f"합계 {sum(m.cost for m in o.matches):.2f}분")

## 4. 최적해가 정말 최적인지 확인합니다

`optimal_match` 는 `scipy` 의 헝가리안 구현을 부릅니다.
그 안을 들여다보지 않고 믿을 것인지, 아니면 확인할 것인지가 이 절의 질문입니다.

확인하는 방법은 단순합니다. 승객을 차량에 배정하는 **모든 경우를 세어 보고**
그중 합이 가장 작은 것을 고릅니다. 5×5 라면 120가지뿐이라 눈 깜짝할 새에 끝납니다.

In [ ]:
from itertools import permutations


def brute_force(costs):
    """모든 배정을 세어 보고 합이 가장 작은 것을 돌려줍니다.

    승객 수와 차량 수가 같은 정사각 행렬만 다룹니다.
    n! 가지를 전부 보므로 8×8 을 넘기면 쓸 수 없습니다.
    그래서 실전용이 아니라 정답지용입니다.
    """
    n = costs.shape[0]
    best_order, best_total = None, float("inf")
    for order in permutations(range(n)):
        total = sum(costs[i, order[i]] for i in range(n))
        if total < best_total:
            best_order, best_total = order, total
    return best_order, best_total

무작위 행렬 200개를 만들어 둘을 맞춰 봅니다.
3장에서 다익스트라를 NetworkX 와 30쌍 맞춰 본 것과 같은 일입니다.

In [ ]:
rng = np.random.default_rng(42)

banner("헝가리안 vs 완전탐색 (5×5, 200회)")
worst = 0.0
mismatch = 0
for _ in range(200):
    m = rng.uniform(1, 30, size=(5, 5))
    _, brute_total = brute_force(m)
    hung_total = sum(x.cost for x in optimal_match(m).matches)
    gap = abs(brute_total - hung_total)
    worst = max(worst, gap)
    mismatch += gap > 1e-9

expect("합이 다른 경우", mismatch, 0)
print(f"    최대 오차 {worst:.2e}")

전부 같습니다. 이제 헝가리안을 믿고 쓸 수 있습니다.

반대로 탐욕 배차를 같은 방식으로 재 보면 얼마나 손해인지 나옵니다.

In [ ]:
banner("탐욕 vs 최적 (5×5, 200회)")
losses = []
for _ in range(200):
    m = rng.uniform(1, 30, size=(5, 5))
    _, best = brute_force(m)
    greedy_total = sum(x.cost for x in greedy_match(m).matches)
    losses.append(greedy_total / best - 1)

print(f"탐욕이 더 쓴 시간  평균 {np.mean(losses):.1%}, 최악 {np.max(losses):.1%}")
print(f"탐욕이 최적과 같았던 비율  {np.mean(np.array(losses) < 1e-9):.0%}")

## 5. 완전탐색을 쓸 수 없는 이유

정답지는 작은 문제에서만 만들 수 있습니다. 경우의 수가 계승으로 늘어나기 때문입니다.

In [ ]:
import math

import pandas as pd

rows = [{"n": n, "경우의 수": math.factorial(n)} for n in [3, 5, 8, 10, 12, 15, 20]]
table = pd.DataFrame(rows)
table["초 (1초에 100만 가지)"] = (table["경우의 수"] / 1e6).round(1)
table

승객 20명이면 경우의 수가 2조를 넘습니다.
실제 시뮬레이션은 1분마다 수십 명을 배차하므로 완전탐색은 처음부터 불가능합니다.
헝가리안은 같은 답을 n³ 시간에 냅니다.

## 6. 얼마나 빠른가

In [ ]:
import time

banner("크기별 실행시간")
for n in [5, 20, 100, 300]:
    m = rng.uniform(1, 30, size=(n, n))
    t0 = time.perf_counter()
    optimal_match(m)
    hung = time.perf_counter() - t0

    t0 = time.perf_counter()
    greedy_match(m)
    greedy = time.perf_counter() - t0

    print(f"{n:4d}×{n:<4d}  헝가리안 {hung * 1000:7.2f} ms   탐욕 {greedy * 1000:7.2f} ms")

문제가 커지면 탐욕이 오히려 느려집니다.
탐욕은 파이썬 반복문이고 헝가리안은 컴파일된 코드이기 때문입니다.
**더 좋은 답을 더 빨리 냅니다.** 탐욕을 쓸 이유가 없습니다.

## 7. 비용을 무엇으로 잴 것인가 (교재 10.6)

지금까지 비용은 직선거리를 평균 속도로 나눈 값이었습니다.
강 건너 차를 가깝다고 잘못 고를 수 있습니다. 도로망 위 실제 소요시간과 비교해 봅니다.

In [ ]:
from smartmob.data import load_road_graph
from smartmob.teaching.dispatch import cost_matrix_from_router

G = load_road_graph("hanam", modes=("drive",))

straight = cost_matrix(passengers, vehicles)
road = cost_matrix_from_router(passengers, vehicles, G)

print("직선거리 기준")
print(np.round(straight, 2))
print("\n도로망 기준")
print(np.round(road, 2))

print("\n같은 배정을 고르는가:",
      [m.vehicle for m in optimal_match(straight).matches]
      == [m.vehicle for m in optimal_match(road).matches])

## 8. 빈칸

### 8.1 탐욕의 처리 순서

`greedy_match` 는 `order` 로 처리 순서를 바꿀 수 있습니다.
순서를 무작위로 바꿔 100번 돌려, 합계가 가장 좋았을 때와 나빴을 때의 차이를 구합니다.
먼저 부른 사람부터 처리하는 것이 좋은 규칙인지 한 줄로 적습니다.

In [ ]:
order_best = None       # 순서를 바꿔 얻은 가장 좋은 합계 (분)
order_worst = None      # 가장 나빴던 합계 (분)

banner("빈칸 8.1")
todo("가장 좋았던 합계", order_best)
todo("가장 나빴던 합계", order_worst)

### 8.2 승객과 차량 수가 다를 때

승객 5명에 차량 3대인 행렬을 만들어 `optimal_match` 를 돌립니다.
배차받지 못한 승객이 `unmatched` 에 들어옵니다.
누가 남는지, 그것이 공평한지 두 줄로 적습니다.

In [ ]:
unmatched_count = None      # 배차받지 못한 승객 수

banner("빈칸 8.2")
todo("배차받지 못한 승객", unmatched_count)

### 8.3 도로망 기준이 답을 바꾸는 경우

승객과 차량 위치를 바꿔 가며, 직선거리 기준과 도로망 기준이 **다른 배정** 을 내는 경우를 하나 찾습니다.
하남은 한강과 산이 있어 그런 자리가 있습니다.

In [ ]:
found_case = None       # 배정이 갈린 (passengers, vehicles) 한 쌍

banner("빈칸 8.3")
todo("배정이 갈린 사례", found_case, fmt=lambda x: "찾았습니다")

## 정리

- 배차는 할당 문제입니다. 탐욕은 사람이 하는 방식이고, 헝가리안은 전체 합을 최소화합니다
- 작은 문제에서 완전탐색으로 정답지를 만들면, 남의 구현을 믿지 않고 검증할 수 있습니다
- 완전탐색은 n! 이라 정답지로만 쓰고, 실전은 n³ 인 헝가리안을 씁니다
- 헝가리안은 탐욕보다 좋은 답을 더 빨리 냅니다
- 비용을 직선거리로 재면 강 건너 차를 고를 수 있습니다
- 11장 실습에서는 이 배차를 1분마다 부르는 루프를 직접 짭니다